In [1]:
import pandas_datareader as pdr
from datetime import datetime
import wbgapi as wb

def get_monthly_quarterly_trade():
    """Get monthly/quarterly import/export data"""
    
    # US Monthly Trade Data (FRED)
    start = datetime(2020, 1, 1)
    end = datetime(2024, 12, 28)
    
    us_trade = {
        'Exports': pdr.DataReader('BOPGEXP', 'fred', start, end),      # Monthly exports
        'Imports': pdr.DataReader('BOPGIMP', 'fred', start, end),      # Monthly imports
        'Trade_Balance': pdr.DataReader('BOPGTB', 'fred', start, end) # Trade balance
    }
    
    return us_trade

# Get monthly US trade data
monthly_trade = get_monthly_quarterly_trade()
print("US Monthly Trade Data:")
print(monthly_trade['Exports'].tail())
print(monthly_trade['Imports'].tail())

/Users/quanghung20gg/code/wtf/.venv/lib/python3.12/site-packages/wbgapi/__init__.py:639: SyntaxWarning: invalid escape sequence '\w'
  pattern = '(?<!\w).{{0,{len}}}{term}.{{0,{len}}}(?!\w)'.format(term=re.escape(q), len=padding)


US Monthly Trade Data:
            BOPGEXP
DATE               
2024-08-01   177954
2024-09-01   175484
2024-10-01   170941
2024-11-01   176509
2024-12-01   170476
            BOPGIMP
DATE               
2024-08-01   275071
2024-09-01   283206
2024-10-01   271029
2024-11-01   282510
2024-12-01   293299


In [21]:
import pandas as pd

def convert_to_long_format(data_dict, indicator_metadata):
    """
    Convert wide format data to long format with datetime, duration, indicator_name, country, value, unit
    
    Args:
        data_dict: Dictionary with indicator names as keys and DataFrames as values
        indicator_metadata: Dictionary mapping indicator names to (code, divisor, unit)
    """
    all_records = []
    
    for indicator_name, df in data_dict.items():
        # Get the unit for this indicator
        unit = indicator_metadata[indicator_name][2]
        
        # Reset index to make country a column
        df_reset = df.reset_index()
        df_reset = df_reset.rename(columns={'economy': 'country'})
        
        # Melt to convert years from columns to rows
        df_long = pd.melt(
            df_reset,
            id_vars=['country'],
            var_name='duration',
            value_name='value'
        )
        
        # Extract year from column names (e.g., 'YR2018' -> 2018)
        df_long['duration'] = df_long['duration'].str.replace('YR', '').astype(int)
        
        # Convert year to datetime (January 1st of each year)
        df_long['datetime'] = pd.to_datetime(df_long['duration'], format='%Y')
        df_long['duration'] = 'annually'
        
        # Add metadata columns
        df_long['indicator_name'] = indicator_name
        df_long['unit'] = unit
        
        # Reorder columns
        df_long = df_long[['datetime', 'duration', 'indicator_name', 'country', 'value', 'unit']]
        
        all_records.append(df_long)
    
    # Combine all indicators
    result = pd.concat(all_records, ignore_index=True)
    
    return result

In [22]:
def get_annual_economic_data(countries, start_year=2010, end_year=2024):
    """
    Get GDP, GNI, Trade, Unemployment for all countries (2010-2025)
    Note: 2025 data likely incomplete
    """
    
    indicators = {
        'GDP': ('NY.GDP.MKTP.CD', 1e9, 'Billion USD'),
        'GDP_Growth_%': ('NY.GDP.MKTP.KD.ZG', 1, 'Percentage'),
        'GNI': ('NY.GNP.MKTP.CD', 1e9, 'Billion USD'),
        'GNI_per_capita': ('NY.GNP.PCAP.CD', 1, 'USD'),
        'Exports_%_GDP': ('NE.EXP.GNFS.ZS', 1, 'Percentage'),
        'Imports_%_GDP': ('NE.IMP.GNFS.ZS', 1, 'Percentage'),
        'Unemployment_%': ('SL.UEM.TOTL.ZS', 1, 'Percentage'),
        'Lending_Rate_%': ('FR.INR.LEND', 1, 'Percentage'),
        'Inflation_%': ('FP.CPI.TOTL.ZG', 1, 'Percentage')
    }
    
    all_data = {}
    time_range = range(start_year, end_year + 1)
    
    print(f"Downloading Annual Economic Data ({start_year}-{end_year})...")
    for indicator_name, (code, divisor, unit) in indicators.items():
        try:
            df = wb.data.DataFrame(code, countries, time=time_range)
            df = df / divisor
            
            all_data[indicator_name] = df
            print(f"  ✓ {indicator_name}")
        except Exception as e:
            print(f"  ✗ {indicator_name}: {str(e)[:50]}")

    df_long = convert_to_long_format(all_data, indicators)
    
    return df_long

countries = [
    'USA', 'CHN', 'JPN', 'DEU', 'IND', 'VNM', 
    'GBR', 'KOR', 'THA', 'SGP', 'FRA', 
]


In [23]:
data_vn = get_annual_economic_data('VNM', 2018)
data_vn

  ✓ GDP
  ✓ GDP_Growth_%
  ✓ GNI
  ✓ GNI_per_capita
  ✓ Exports_%_GDP
  ✓ Imports_%_GDP
  ✓ Unemployment_%
  ✓ Lending_Rate_%
  ✓ Inflation_%


,datetime,duration,indicator_name,country,value,unit
0,2018-01-01,annually,GDP,VNM,310.106478,Billion USD
1,2019-01-01,annually,GDP,VNM,334.365270,Billion USD
2,2020-01-01,annually,GDP,VNM,346.615739,Billion USD
3,2021-01-01,annually,GDP,VNM,366.474753,Billion USD
4,2022-01-01,annually,GDP,VNM,413.445231,Billion USD
...,...,...,...,...,...,...
58,2020-01-01,annually,Inflation_%,VNM,3.220934,Percentage
59,2021-01-01,annually,Inflation_%,VNM,1.834716,Percentage
60,2022-01-01,annually,Inflation_%,VNM,3.156507,Percentage
61,2023-01-01,annually,Inflation_%,VNM,3.252893,Percentage


In [10]:
data_vn['Exports_%_GDP']

,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023,YR2024
economy,,,,,,,
VNM,84.423456,85.157587,84.381598,93.850207,93.420129,86.467419,90.154221


In [36]:
def convert_treasury_to_long_format(treasury_df, country='USA'):
    """
    Convert treasury DataFrame to long format
    """
    all_records = []
    
    # Reset index to make DATE a column
    df_reset = treasury_df.reset_index()
    df_reset = df_reset.rename(columns={'DATE': 'datetime'})
    
    # Melt to convert treasury types from columns to rows
    df_long = pd.melt(
        df_reset,
        id_vars=['datetime'],
        var_name='indicator_name',
        value_name='value'
    )
    
    # Add metadata columns
    df_long['duration'] = 'monthly'
    df_long['country'] = country
    df_long['unit'] = 'Percentage'
    
    # Map indicator names to more descriptive names
    df_long['indicator_name'] = df_long['indicator_name'].map({
        '2Y': 'USA_2Y',
        '10Y': 'USA_10Y',
    })
    
    # Reorder columns
    df_long = df_long[['datetime', 'duration', 'indicator_name', 'country', 'value', 'unit']]
    
    return df_long

def get_daily_us_treasury(start_year=2010):
    """Get daily US 2Y and 10Y Treasury yields from 2010-2025"""
    
    start = datetime(start_year, 1, 1)
    end = datetime.now()
    
    print("\nDownloading Daily US Treasury Rates (2010-2025)...")
    
    treasury = pd.DataFrame()
    
    try:
        treasury['2Y'] = pdr.DataReader('DGS2', 'fred', start, end)['DGS2']
        print("  ✓ US 2-Year Treasury (Daily)")
    except Exception as e:
        print(f"  ✗ US 2Y: {e}")
    
    try:
        treasury['10Y'] = pdr.DataReader('DGS10', 'fred', start, end)['DGS10']
        print("  ✓ US 10-Year Treasury (Daily)")
    except Exception as e:
        print(f"  ✗ US 10Y: {e}")

    treasury = treasury.resample('M').mean()

    # Convert to long format
    treasury_long = convert_treasury_to_long_format(treasury, country='USA')
    
    return treasury_long

In [37]:
daily_us_treasury = get_daily_us_treasury(2018)


  ✓ US 2-Year Treasury (Daily)
  ✓ US 10-Year Treasury (Daily)


/var/folders/y0/d5yxmby14p1b51jsfj3ml2vr0000gn/T/ipykernel_76952/3235208882.py:57: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  treasury = treasury.resample('M').mean()


In [38]:
daily_us_treasury

,datetime,duration,indicator_name,country,value,unit
0,2018-01-31,monthly,USA_2Y,USA,2.031905,Percentage
1,2018-02-28,monthly,USA_2Y,USA,2.175263,Percentage
2,2018-03-31,monthly,USA_2Y,USA,2.275714,Percentage
3,2018-04-30,monthly,USA_2Y,USA,2.384286,Percentage
4,2018-05-31,monthly,USA_2Y,USA,2.511818,Percentage
...,...,...,...,...,...,...
187,2025-08-31,monthly,USA_10Y,USA,4.264762,Percentage
188,2025-09-30,monthly,USA_10Y,USA,4.120476,Percentage
189,2025-10-31,monthly,USA_10Y,USA,4.061818,Percentage
190,2025-11-30,monthly,USA_10Y,USA,4.093889,Percentage


In [ ]:
def convert_bond_yields_to_long_format(bond_yields_dict):
    """
    Convert bond yields dictionary to long format
    """
    all_records = []
    
    # Mapping from key names to (country, indicator_name)
    mapping = {
        'Japan_10Y': ('JPN', 'Government_Bond_10Y'),
        'Korea_10Y': ('KOR', 'Government_Bond_10Y')
    }
    
    for key, df in bond_yields_dict.items():
        if df.empty:
            continue
            
        country, indicator_name = mapping.get(key, (key, 'Bond_Yield'))
        
        # Reset index to make DATE a column
        df_reset = df.reset_index()
        df_reset = df_reset.rename(columns={'DATE': 'datetime'})
        
        # Get the value column (first non-DATE column)
        value_col = df_reset.columns[1]
        
        # Create long format
        df_long = pd.DataFrame({
            'datetime': df_reset['datetime'],
            'duration': 'monthly',
            'indicator_name': indicator_name,
            'country': country,
            'value': df_reset[value_col],
            'unit': 'Percentage'
        })
        
        all_records.append(df_long)
    
    # Combine all bond yields
    if all_records:
        result = pd.concat(all_records, ignore_index=True)
        return result
    else:
        return pd.DataFrame()

def get_monthly_bond_yields(start_year=2018):
    """Get monthly 10Y government bond yields for Japan and Korea"""
    
    start = datetime(start_year, 1, 1)
    end = datetime.now()
    
    print("\nDownloading Monthly Bond Yields (2010-2025)...")
    
    bond_yields = {}
    
    # Japan 10Y
    try:
        bond_yields['Japan_10Y'] = pdr.DataReader('IRLTLT01JPM156N', 'fred', start, end)
        print("  ✓ Japan 10-Year (Monthly)")
    except Exception as e:
        print(f"  ✗ Japan: {e}")
    
    # Korea 10Y
    try:
        bond_yields['Korea_10Y'] = pdr.DataReader('IRLTLT01KRM156N', 'fred', start, end)
        print("  ✓ South Korea 10-Year (Monthly)")
    except Exception as e:
        print(f"  ✗ Korea: {e}")
    
    # Convert to long format
    bond_yields_long = convert_bond_yields_to_long_format(bond_yields)
    
    return bond_yields_long

data = get_monthly_bond_yields()


  ✓ Japan 10-Year (Monthly)
  ✓ South Korea 10-Year (Monthly)
  ✗ China: Unable to read URL: https://fred.stlouisfed.org/graph/fredgraph.csv?id=IRLTLT01CNM156N
Response Text:
b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n    <meta charset="utf-8">\r\n    <meta http-equiv="X-UA-Compatible" content="IE=edge">\r\n    <meta name="viewport" content="width=device-width, initial-scale=1">\r\n    <title>Error - St. Louis Fed</title>\r\n    <meta name="description" content="">\r\n    <meta name="keywords" content="">    \r\n    <link rel="stylesheet" type="text/css" href="/assets/bootstrap/dist/css/bootstrap.min.css">\r\n    <link rel="stylesheet" type="text/css" href="/css/home.min.css?1553087253">\r\n    <link rel="stylesheet" type="text/css" href="/assets/fontawesome-free/css/all.min.css">\r\n    <link rel="stylesheet" type="text/css" href="/assets/select2/dist/css/select2.min.css">\r\n    <style>p {\r\n        margin-bottom: 1.5em;\r\n    }</style>\r\n</head>\r\n<body>\r\n<link rel="p

In [32]:
data

{'Japan_10Y':             IRLTLT01JPM156N
 DATE                       
 2018-01-01            0.080
 2018-02-01            0.045
 2018-03-01            0.045
 2018-04-01            0.050
 2018-05-01            0.030
 ...                     ...
 2025-07-01            1.545
 2025-08-01            1.600
 2025-09-01            1.645
 2025-10-01            1.655
 2025-11-01            1.805
 
 [95 rows x 1 columns],
 'Korea_10Y':             IRLTLT01KRM156N
 DATE                       
 2018-01-01            2.626
 2018-02-01            2.770
 2018-03-01            2.708
 2018-04-01            2.655
 2018-05-01            2.757
 ...                     ...
 2025-07-01            2.840
 2025-08-01            2.815
 2025-09-01            2.849
 2025-10-01            2.933
 2025-11-01            3.248
 
 [95 rows x 1 columns]}